# v26d — fit the detectors on train, select on dev, hash and freeze (CPU, any kernel)

Pre-registered detectors (README "Detectors"). **This notebook can only read train / dev / selection rows** (I3); test rows are
never loaded into a fit. Every detector is saved to `artifacts/v26/<model>/` and hashed into `artifacts/v26_frozen.json`.
Once v26e has run (it writes `results/v26/_evaluated.lock`), this notebook refuses to re-freeze.

| id | family | features | label (train) | selection (dev) |
|---|---|---|---|---|
| D2 | presence | last-token residual, 1 layer | attack vs twin + look-alike + reader | AUROC attack vs negatives |
| D3 | presence | Δ = end − anchor, 1 layer | same | same |
| D4 | routing | 1 − mean attention focus on the task over selected heads | none (heads from the 60 selection rows) | — |
| D5 | routing + breach | per-head attention-competition ratio, all heads | existence: attack vs negatives; breach: success vs resisted | AUROC each |
| D6 | breach | last-token residual, 1 layer | successful attack vs everything else | AUROC success vs rest |

Descriptive extras: D2s / D3s (standard labels: attack vs twin only), two shortcut baselines (I4), noise refits (I5).

In [ ]:
import os, json, hashlib, time
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed

MODELS = ["qwen3-4b", "llama3.2-3b", "llama3-8b"]
MAIN = {"qwen3-4b", "llama3.2-3b"}
C_GRID, C_GRID_L1 = [1e-3, 1e-2, 1e-1, 1.0], [1e-2, 1e-1, 1.0]
TIE, K_SIGMA, FALLBACK_HEADS = 0.005, 4.0, 10
ART, RES = "artifacts/v26", "results/v26"
os.makedirs(ART, exist_ok=True); os.makedirs(RES, exist_ok=True)
if os.path.exists(f"{RES}/_evaluated.lock"):
    raise RuntimeError("v26e has already evaluated the frozen detectors. Re-freezing is not allowed (pre-registration).")
N_JOBS = min(8, os.cpu_count() or 2)

def sha(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
def auc(y, s): return float(roc_auc_score(y, s))

def fit_lr(X, y, C, penalty="l2"):
    sc = StandardScaler().fit(X)
    lr = LogisticRegression(C=C, penalty=penalty, class_weight="balanced", max_iter=3000,
                            solver="liblinear" if penalty == "l1" else "lbfgs").fit(sc.transform(X), y)
    return sc, lr

def score(sc, lr, X): return lr.decision_function(sc.transform(X))

def pick(table, nl):
    """table: list of (layer, C, dev_auc). Best AUROC; ties within TIE go to the layer closest to the middle, then the smaller C."""
    best = max(t[2] for t in table)
    cand = [t for t in table if t[2] >= best - TIE]
    return min(cand, key=lambda t: (abs(t[0] - (nl - 1) / 2), t[1]))

def save(model, name, **arrs):
    p = f"{ART}/{model}/{name}.npz"; np.savez(p, **arrs); return p

FROZEN = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "models": {}}

## Fit, select, freeze — one model at a time

In [ ]:
for model in MODELS:
    D = f"representation_cache/v26/{model}"
    if not os.path.exists(f"{D}/meta.csv"):
        print(f"[{model}] not extracted -- skipped"); continue
    os.makedirs(f"{ART}/{model}", exist_ok=True)
    t0 = time.time()
    M = pd.read_csv(f"{D}/meta.csv", keep_default_na=False)
    M["success"] = M.success.astype(str).str.lower().eq("true")
    man = json.load(open(f"{D}/manifest.json")); NL, NH = man["layers"], man["heads"]
    attn = np.load(f"{D}/attn.npy")
    rho = attn[..., 1] / (attn[..., 1] + attn[..., 2] + 1e-8)          # (N, layers, heads): focus on the task vs the data
    info = {"rows_sha256": man["rows_sha256"], "files": {}, "selection": {}}

    # ---- D4: Attention Tracker head selection on the 60 synthetic rows (no v26 label) ----
    nrm, atk = (M.tier == "atsel_normal").values, (M.tier == "atsel_attack").values
    mu_n, mu_a = rho[nrm].mean(0), rho[atk].mean(0); sd_n, sd_a = rho[nrm].std(0), rho[atk].std(0)
    crit = mu_n - mu_a - K_SIGMA * (sd_n + sd_a)
    heads = np.argwhere(crit > 0)
    how = f"criterion mu_n - mu_a - {K_SIGMA}(sd_n + sd_a) > 0"
    if len(heads) < 3:                                       # pre-specified fallback (README run notes)
        z = ((mu_n - mu_a) / (sd_n + sd_a + 1e-8)).ravel()
        heads = np.array(np.unravel_index(np.argsort(-z)[:FALLBACK_HEADS], (NL, NH))).T
        how = f"fallback: top {FALLBACK_HEADS} heads by (mu_n - mu_a)/(sd_n + sd_a) (criterion selected < 3)"
    info["files"]["D4"] = save(model, "D4", heads=heads)
    info["selection"]["D4"] = dict(n_heads=int(len(heads)), how=how)
    print(f"[{model}] D4: {len(heads)} heads ({how})")

    if model not in MAIN:
        print(f"[{model}] check model: test split only -> D4 only (no training rows)")
        FROZEN["models"][model] = info; continue

    TR = np.where(M.split == "train")[0]; DV = np.where(M.split == "dev")[0]
    assert not M.split.iloc[np.r_[TR, DV]].isin(["test"]).any()          # I3
    att = (M.tier == "attack").values; neg = M.tier.isin(["twin", "lookalike", "reader"]).values
    twin = (M.tier == "twin").values; succ = (M.success & att).values
    if succ[TR].sum() < 20 or (att & ~succ)[TR].sum() < 20:
        raise RuntimeError(f"{model}: too few successful ({succ[TR].sum()}) or resisted ({(att & ~succ)[TR].sum()}) "
                           "training attacks to fit the breach detectors. Stop and send Claude this output.")

    end = np.load(f"{D}/end.npy", mmap_mode="r"); anc = np.load(f"{D}/anchor.npy", mmap_mode="r")
    TRDV = np.r_[TR, DV]

    def layer_feats(li, kind):
        e = np.asarray(end[TRDV, li, :], dtype=np.float32)
        return e if kind == "end" else e - np.asarray(anc[TRDV, li, :], dtype=np.float32)

    specs = {   # name: (feature kind, train mask, label, dev mask, dev label)
        "D2":  ("end",   att | neg,  att,  att | neg,  att),
        "D3":  ("delta", att | neg,  att,  att | neg,  att),
        "D6":  ("end",   att | neg,  succ, att | neg,  succ),
        "D2s": ("end",   att | twin, att,  att | twin, att),
        "D3s": ("delta", att | twin, att,  att | twin, att),
    }
    pos_tr = {i: k for k, i in enumerate(TRDV)}
    trk = np.array([pos_tr[i] for i in TR]); dvk = np.array([pos_tr[i] for i in DV])

    def sweep_layer(li):
        out = []
        X = {k: layer_feats(li, k) for k in ("end", "delta")}
        for name, (kind, mtr, ytr, mdv, ydv) in specs.items():
            a, b = trk[mtr[TR]], dvk[mdv[DV]]
            for C in C_GRID:
                sc, lr = fit_lr(X[kind][a], ytr[TR][mtr[TR]].astype(int), C)
                out.append((name, li, C, auc(ydv[DV][mdv[DV]].astype(int), score(sc, lr, X[kind][b]))))
        return out
    res = [r for rr in Parallel(n_jobs=N_JOBS)(delayed(sweep_layer)(li) for li in range(NL)) for r in rr]
    sweep = pd.DataFrame(res, columns=["detector", "layer", "C", "dev_auc"])
    sweep.to_csv(f"{RES}/{model}_dev_sweep.csv", index=False)

    rng = np.random.default_rng(26)
    for name, (kind, mtr, ytr, mdv, ydv) in specs.items():
        tbl = [tuple(r) for r in sweep[sweep.detector == name][["layer", "C", "dev_auc"]].itertuples(index=False)]
        li, C, dauc = pick(tbl, NL)
        X = layer_feats(li, kind); a = trk[mtr[TR]]; y = ytr[TR][mtr[TR]].astype(int)
        sc, lr = fit_lr(X[a], y, C)
        info["files"][name] = save(model, name, kind=kind, layer=li, C=C, mean=sc.mean_, scale=sc.scale_,
                                   coef=lr.coef_[0], intercept=lr.intercept_)
        info["selection"][name] = dict(layer=int(li), C=float(C), dev_auc=float(dauc), n_train=int(len(a)))
        if name in ("D2", "D3", "D6"):                      # I5 noise test: random Gaussian FEATURES, real labels, same C
            seed = 2600 + ["D2", "D3", "D6"].index(name)    # v26e regenerates the same arrays from this seed
            Z = np.random.default_rng(seed).standard_normal((len(M), X.shape[1]), dtype=np.float32)
            sc_n, lr_n = fit_lr(Z[TR][mtr[TR]], y, C)
            info["files"][name + "_noise"] = save(model, name + "_noise", kind="noise", noise_seed=seed, layer=li, C=C,
                                                  mean=sc_n.mean_, scale=sc_n.scale_, coef=lr_n.coef_[0],
                                                  intercept=lr_n.intercept_, dim=X.shape[1], n_rows=len(M))
            del Z
        print(f"[{model}] {name}: layer {li}/{NL-1}, C={C:g}, dev AUROC {dauc:.4f}")

    # ---- D5: BASIS-style existence x breach on per-head competition ratios ----
    F = rho.reshape(len(M), -1)
    best = {}
    for part, mtr, ytr, mdv, ydv in (("exist", att | neg, att, att | neg, att), ("breach", att, succ, att, succ)):
        tab = []
        for C in C_GRID_L1:
            sc, lr = fit_lr(F[TR][mtr[TR]], ytr[TR][mtr[TR]].astype(int), C, penalty="l1")
            tab.append((C, auc(ydv[DV][mdv[DV]].astype(int), score(sc, lr, F[DV][mdv[DV]])), sc, lr))
        top = max(t[1] for t in tab)
        C, dauc, sc, lr = min([t for t in tab if t[1] >= top - TIE], key=lambda t: t[0])   # ties -> stronger regularisation
        best[part] = (C, dauc, sc, lr)
        print(f"[{model}] D5 {part}: C={C:g}, dev AUROC {dauc:.4f}, non-zero heads {(lr.coef_ != 0).sum()}")
    (Ce, ae, sce, lre), (Cb, ab, scb, lrb) = best["exist"], best["breach"]
    info["files"]["D5"] = save(model, "D5", mean_e=sce.mean_, scale_e=sce.scale_, coef_e=lre.coef_[0], b_e=lre.intercept_,
                               mean_b=scb.mean_, scale_b=scb.scale_, coef_b=lrb.coef_[0], b_b=lrb.intercept_)
    info["selection"]["D5"] = dict(C_exist=Ce, dev_auc_exist=ae, C_breach=Cb, dev_auc_breach=ab,
                                   score="p_exist * p_breach (single-score adaptation of BASIS's cascade)")

    # ---- I4 shortcut baselines (presence labels) ----
    geo = {"length": M[["ntok"]].assign(d=M.data_e - M.data_s).values.astype(float),
           "geometry": M[["data_s", "data_e", "ntok"]].values.astype(float)}
    for name, G in geo.items():
        a = TR[(att | neg)[TR]]
        sc, lr = fit_lr(G[a], att[a].astype(int), 1.0)
        info["files"]["SC_" + name] = save(model, "SC_" + name, mean=sc.mean_, scale=sc.scale_, coef=lr.coef_[0],
                                           intercept=lr.intercept_)
    FROZEN["models"][model] = info
    print(f"[{model}] done in {(time.time()-t0)/60:.1f} min")

## Hash and write the manifest

In [ ]:
for m, info in FROZEN["models"].items():
    info["sha256"] = {k: sha(p) for k, p in info["files"].items()}
json.dump(FROZEN, open("artifacts/v26_frozen.json", "w"), indent=1, default=float)
print("froze:", {m: sorted(i["files"]) for m, i in FROZEN["models"].items()})
print("NEXT: v26e_evaluate.ipynb (needs results/v26/audit_result.json from v26c Part 2)")